## Insertion of Dimension Tables

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

#### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    col, date_add, lit, dayofweek, dayofmonth,
    weekofyear, month, quarter, year, last_day,
    date_format, when, explode, sequence, to_date
)
from pyspark.sql.types import DateType

#### DIM_DATE TABLE CREATION

In [0]:
# Generate a date sequence from 2016-01-01 to 2018-12-31
df_date = spark.sql("""
    SELECT explode(sequence(
        to_date('2016-01-01'),
        to_date('2018-12-31'),
        interval 1 day
    )) AS full_date
""")

# Build all calendar attributes
df_dim_date = (
    df_date
    .withColumn("date_key",
        (year("full_date") * 10000 +
         month("full_date") * 100 +
         dayofmonth("full_date")).cast("integer"))
    .withColumn("day_of_week",
        date_format("full_date", "EEEE"))
    .withColumn("day_of_month",
        dayofmonth("full_date"))
    .withColumn("week_of_year",
        weekofyear("full_date"))
    .withColumn("month_number",
        month("full_date"))
    .withColumn("month_name",
        date_format("full_date", "MMMM"))
    .withColumn("quarter",
        when(month("full_date").isin(1,2,3), "Q1")
        .when(month("full_date").isin(4,5,6), "Q2")
        .when(month("full_date").isin(7,8,9), "Q3")
        .otherwise("Q4"))
    .withColumn("year",
        year("full_date"))
    .withColumn("is_weekend",
        dayofweek("full_date").isin(1, 7))
    .withColumn("is_month_end",
        col("full_date") == last_day("full_date"))
    .withColumn("is_quarter_end",
        when(
            (month("full_date").isin(3, 6, 9, 12)) &
            (col("full_date") == last_day("full_date")),
            True).otherwise(False))
    .withColumn("is_black_friday",
        col("full_date") == lit("2017-11-24").cast(DateType()))
    .withColumn("season",
        # Brazilian seasons (Southern Hemisphere)
        when(month("full_date").isin(12, 1, 2), "Summer")
        .when(month("full_date").isin(3, 4, 5), "Autumn")
        .when(month("full_date").isin(6, 7, 8), "Winter")
        .otherwise("Spring"))
)

# Sanity check
print("Total days generated:", df_dim_date.count())
df_dim_date.show(5, truncate=False)

In [0]:
df_dim_date = df_dim_date.withColumn(
    "month_year",
    date_format("full_date", "MMM yyyy")  # Sep 2016, Oct 2016 etc.
)
df_dim_date.show(5, truncate=False)

In [0]:
(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.dim_date")
)

print("dim_date written successfully")

#### DIM_CUSTOMERS TABLE CREATION


In [0]:
df_slv_customers = spark.table("olist_ecommerce_project.silver.slv_customers")

df_dim_customers = (
    df_slv_customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_state",
        "customer_city",
        "customer_zip_code_prefix"
    )
    .distinct()
)

print("dim_customers rows:", df_dim_customers.count())
df_dim_customers.show(5, truncate=False)

# Write to Gold
(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.dim_customers")
)

print("dim_customers written successfully")

#### DIM_PRODUCTS TABLE CREATION

In [0]:
df_slv_products = spark.table("olist_ecommerce_project.silver.slv_products")

df_dim_products = (
    df_slv_products
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

print("dim_products rows:", df_dim_products.count())
df_dim_products.show(5, truncate=False)

# Write to Gold
(
    df_dim_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.dim_products")
)

print("dim_products written successfully")

#### DIM_SELLERS TABLE CREATION

In [0]:
df_slv_sellers = spark.table("olist_ecommerce_project.silver.slv_sellers")

df_dim_sellers = (
    df_slv_sellers
    .select(
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    )
    .drop("_ingestion_timestamp")
)

print("dim_sellers rows:", df_dim_sellers.count())
df_dim_sellers.show(5, truncate=False)

# Write to Gold
(
    df_dim_sellers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.dim_sellers")
)

print("dim_sellers written successfully")

#### DIM_GEOGRAPHY TABLE CREATION

In [0]:
df_slv_geolocation = spark.table("olist_ecommerce_project.silver.slv_geolocation")

df_dim_geography = (
    df_slv_geolocation
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "geolocation_lat",
        "geolocation_lng"
    )
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("geolocation_city", "city")
    .withColumnRenamed("geolocation_state", "state")
    .withColumnRenamed("geolocation_lat", "latitude")
    .withColumnRenamed("geolocation_lng", "longitude")
)

print("dim_geography rows:", df_dim_geography.count())
df_dim_geography.show(5, truncate=False)

# Write to Gold
(
    df_dim_geography.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.dim_geography")
)

print("dim_geography written successfully")